---
## Section 1 — Imports & Configuration

All tunable parameters are consolidated here. Adjust `TRAIN_START_YEAR`,
`TRAIN_END_YEAR`, `VAL_START_YEAR`, `VAL_END_YEAR`, and `FORECAST_END_YEAR` to
shift the time windows without touching any other cell.


In [1]:
df = spark.sql("SELECT * FROM MDMF_COPY_RN.dbo.dim_five_essentials LIMIT 1000")
display(df)

StatementMeta(, d19cdaca-6fa0-468f-918e-5ca30b324312, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bb90f5bb-b5ec-45d0-ab86-d329baccb76a)

In [26]:
# Export ml_enrollment_feature_table_v4 as a single CSV file into Lakehouse Files
from pyspark.sql import functions as F

# 1. Read the feature table
feat_df = spark.read.table("ml_enrollment_feature_table_v4")
print(f"Input rows: {feat_df.count():,}")

# 2. Write to a temporary CSV folder (single partition) under Files
csv_dir = "Files/ml_enrollment_feature_table_v4_csv"

# Clean up any previous export folder
if notebookutils.fs.exists(csv_dir):
    notebookutils.fs.rm(csv_dir, recurse=True)

feat_df.coalesce(1).write.mode("overwrite").option("header", "true").csv(csv_dir)
print(f"Wrote CSV folder: {csv_dir}")

# 3. Find the actual CSV part file and copy it to a single-file path
files = notebookutils.fs.ls(csv_dir)
csv_parts = [f.path for f in files if f.path.endswith(".csv")]

if not csv_parts:
    raise RuntimeError("No CSV part file found in export directory.")

single_csv_path = "Files/ml_enrollment_feature_table_v4.csv"
notebookutils.fs.cp(csv_parts[0], single_csv_path, recurse=False)
print(f"Single CSV file written to: {single_csv_path}")
print("You can now download 'ml_enrollment_feature_table_v4.csv' from the Lakehouse Files pane.")

StatementMeta(, d19cdaca-6fa0-468f-918e-5ca30b324312, 6, Finished, Available, Finished, False)

Input rows: 31,750


Wrote CSV folder: Files/ml_enrollment_feature_table_v4_csv
Single CSV file written to: Files/ml_enrollment_feature_table_v4.csv
You can now download 'ml_enrollment_feature_table_v4.csv' from the Lakehouse Files pane.


In [2]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType
from pyspark.sql.window import Window

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder

import pandas as pd
import numpy as np

# ---------------------------------------------------------------------------
# INPUT TABLES
# ---------------------------------------------------------------------------
CLEAN_TABLE   = "fact_enrollment_annualized_clean"   # cleaned enrollment facts
DIM_TABLE     = "dim_school"                          # school metadata

# OUTPUT TABLES
FEATURE_TABLE = "ml_enrollment_feature_table_v4"     # written by Stage 1
FORECAST_TABLE_SUFFIX = "v3_fixed"                   # forecast table suffix

# ---------------------------------------------------------------------------
# YEAR WINDOWS  — edit these to move the train/val/forecast windows
# ---------------------------------------------------------------------------
TRAIN_START_YEAR    = 2020
TRAIN_END_YEAR      = 2025
VAL_START_YEAR      = 2026
VAL_END_YEAR        = 2026
FORECAST_END_YEAR   = 2027

# ---------------------------------------------------------------------------
# MODEL HYPERPARAMETERS  (fixed best-params from prior grid search)
# ---------------------------------------------------------------------------
BEST_MAX_DEPTH        = 5
BEST_MAX_ITER         = 100
BEST_STEP_SIZE        = 0.1
BEST_SUBSAMPLING_RATE = 0.8

# ---------------------------------------------------------------------------
# MISC
# ---------------------------------------------------------------------------
LABEL_COL  = "ENROLLMENT"
TIME_COL   = "SCHOOL_YEAR"
WEIGHT_COL = "sample_weight"

# Include all grades K–12 in the feature table; entry grades (K, 9) rely more on
# imputed lag/feeder features when they lack internal history.
TARGET_GRADES = ["K", "1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11", "12"]

GRADE_ORDER = {
    "PE": 0, "PK": 1, "K": 2,
    "1":  3, "2":  4, "3":  5, "4":  6,  "5":  7,  "6": 8,
    "7":  9, "8": 10, "9": 11, "10": 12, "11": 13, "12": 14,
}

print("Configuration loaded.")
print(f"  Train  : {TRAIN_START_YEAR} – {TRAIN_END_YEAR}")
print(f"  Val    : {VAL_START_YEAR} – {VAL_END_YEAR}")
print(f"  Forecast year: {FORECAST_END_YEAR}")


StatementMeta(, d19cdaca-6fa0-468f-918e-5ca30b324312, 7, Finished, Available, Finished, False)

Configuration loaded.
  Train  : 2020 – 2025
  Val    : 2026 – 2026
  Forecast year: 2027


In [3]:
df = spark.sql("SELECT * FROM MDMF_COPY_RN.dbo.ml_enrollment_feature_table_v4 LIMIT 1000")
display(df)

StatementMeta(, d19cdaca-6fa0-468f-918e-5ca30b324312, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 06f8124a-103f-4282-8784-1d39834c094b)

---
## Section 2 — Load & Aggregate Enrollment Data

Load `fact_enrollment_annualized_clean`, filter to non-future, non-zero-day rows,
map grades to numeric order, and aggregate to one row per
**school × grade × school year** by counting distinct students.


In [4]:
df_raw = spark.read.table(CLEAN_TABLE)
print(f"Raw rows: {df_raw.count():,}  |  Columns: {len(df_raw.columns)}")

grade_map_expr = F.create_map([F.lit(x) for pair in GRADE_ORDER.items() for x in pair])

df = (
    df_raw
    .filter(F.col("IS_FUTURE_SCHOOL_YEAR") == False)
    .filter(F.col("IS_ZERO_DAY_ENROLLMENT") == False)
    .withColumn("GRADE", F.col("ENTRY_GRADE_LEVEL").cast("string"))
    .withColumn("GRADE_NUMERIC", grade_map_expr[F.col("GRADE")])
    .filter(F.col("GRADE_NUMERIC").isNotNull())
)
print(f"After filtering (no future, no zero-day): {df.count():,}")

df_agg = (
    df
    .groupBy("SCHOOL_KEY", TIME_COL, "GRADE", "GRADE_NUMERIC")
    .agg(F.countDistinct("STUDENT_KEY").alias(LABEL_COL))
)
print(f"Aggregated rows (school × grade × year): {df_agg.count():,}")


StatementMeta(, d19cdaca-6fa0-468f-918e-5ca30b324312, 9, Finished, Available, Finished, False)

Raw rows: 9,875,703  |  Columns: 32


After filtering (no future, no zero-day): 2,940,738


Aggregated rows (school × grade × year): 50,912


---
## Section 3 — Lag Features & Feeder Grade

### Same-grade lags
`SAME_GRADE_LAST_YEAR` and `SAME_GRADE_2YR_AGO` are simply `lag(ENROLLMENT, 1)`
and `lag(ENROLLMENT, 2)` within each school-grade partition ordered by year.
These are purely historical — no leakage.

### Feeder grade (`FEEDER_GRADE_LAST_YEAR`, `FEEDER_GRADE_2YR_AGO`)
> *CSR = Grade N this year ÷ Grade N−1 last year*

So the feeder is grade N−1 at the **same school** in the **prior year**.
The join matches `GRADE_NUMERIC == feeder.GRADE_NUMERIC + 1` and
`SCHOOL_YEAR == feeder.SCHOOL_YEAR + 1`.

`HAS_FEEDER_GRADE` is set **before** the coalesce fallback so it accurately
reflects whether a real feeder was found (not the fallback value).


In [5]:
w_sg = Window.partitionBy("SCHOOL_KEY", "GRADE").orderBy(TIME_COL)

# ── 3a. Same-grade lags ──────────────────────────────────────────────────
df_feat = (
    df_agg
    .withColumn("SAME_GRADE_LAST_YEAR", F.lag(LABEL_COL, 1).over(w_sg))
    .withColumn("SAME_GRADE_2YR_AGO",   F.lag(LABEL_COL, 2).over(w_sg))
)

# ── 3b. Feeder grade lookup ──────────────────────────────────────────────
# Domain definition: feeder = grade N-1 at same school, prior year.
feeder_lookup = (
    df_agg.select(
        F.col("SCHOOL_KEY").alias("_fk_school"),
        F.col("GRADE_NUMERIC").alias("_fk_grade_num"),
        F.col(TIME_COL).alias("_fk_year"),
        F.col(LABEL_COL).alias("FEEDER_GRADE_LAST_YEAR"),
    )
)

df_feat = (
    df_feat.join(
        feeder_lookup,
        on=[
            df_feat["SCHOOL_KEY"]    == feeder_lookup["_fk_school"],
            df_feat["GRADE_NUMERIC"] == feeder_lookup["_fk_grade_num"] + 1,
            df_feat[TIME_COL]        == feeder_lookup["_fk_year"] + 1,
        ],
        how="left",
    )
    .drop("_fk_school", "_fk_grade_num", "_fk_year")
    # HAS_FEEDER_GRADE set BEFORE coalesce so it reflects a real feeder, not fallback
    .withColumn(
        "HAS_FEEDER_GRADE",
        F.when(F.col("FEEDER_GRADE_LAST_YEAR").isNotNull(), F.lit(1)).otherwise(F.lit(0))
    )
    .withColumn(
        "FEEDER_GRADE_LAST_YEAR",
        F.coalesce(F.col("FEEDER_GRADE_LAST_YEAR"), F.col("SAME_GRADE_LAST_YEAR"))
    )
)

# ── 3c. Feeder grade two years ago (BUG MT-1 fix: build what training references) ──
feeder_2yr_lookup = (
    df_agg.select(
        F.col("SCHOOL_KEY").alias("_f2_school"),
        F.col("GRADE_NUMERIC").alias("_f2_grade_num"),
        F.col(TIME_COL).alias("_f2_year"),
        F.col(LABEL_COL).alias("FEEDER_GRADE_2YR_AGO"),
    )
)

df_feat = (
    df_feat.join(
        feeder_2yr_lookup,
        on=[
            df_feat["SCHOOL_KEY"]    == feeder_2yr_lookup["_f2_school"],
            df_feat["GRADE_NUMERIC"] == feeder_2yr_lookup["_f2_grade_num"] + 1,
            df_feat[TIME_COL]        == feeder_2yr_lookup["_f2_year"] + 2,   # 2 years back
        ],
        how="left",
    )
    .drop("_f2_school", "_f2_grade_num", "_f2_year")
    .withColumn(
        "FEEDER_GRADE_2YR_AGO",
        F.coalesce(F.col("FEEDER_GRADE_2YR_AGO"), F.col("SAME_GRADE_2YR_AGO"))
    )
)

print("Lag + feeder features built.")
print(f"  Rows: {df_feat.count():,}")


StatementMeta(, d19cdaca-6fa0-468f-918e-5ca30b324312, 10, Finished, Available, Finished, False)

Lag + feeder features built.


  Rows: 50,912


---
## Section 4 — Cohort Survival Rate (Leak-Free)



### Why it matters
CPS defines CSR as:
> *"CSR = This Year Grade 7 Enrollment ÷ Last Year Grade 6 Enrollment"*

This is a **cross-grade** ratio (feeder → promoted grade). It should **never**
include the current-year label value — that would let the model see the answer.

Both features now contain only information that was available **before** the target year.


In [6]:
# _SR(t) = ENROLLMENT(t) / FEEDER_GRADE_LAST_YEAR(t)  [cross-grade ratio, per domain guide]
# This is the "true" CSR for year t. We compute it but NEVER use it as a feature —
# it contains ENROLLMENT(t) in the numerator and is therefore the label.
# We only use its LAGGED values: SR(t-1), SR(t-2), SR(t-3).

df_feat = df_feat.withColumn(
    "_SR_CURRENT",
    F.when(
        (F.col("FEEDER_GRADE_LAST_YEAR").isNotNull()) & (F.col("FEEDER_GRADE_LAST_YEAR") > 0),
        F.col(LABEL_COL) / F.col("FEEDER_GRADE_LAST_YEAR")
    ).otherwise(F.lit(None))
)

# BUG FE-1 FIX: lag by 1 so COHORT_SURVIVAL_RATE only uses past information
df_feat = df_feat.withColumn(
    "COHORT_SURVIVAL_RATE",
    F.lag("_SR_CURRENT", 1).over(w_sg)
)

# BUG FE-2 FIX: average only over fully lagged SR values (t-1, t-2, t-3)
df_feat = (
    df_feat
    .withColumn("_SR_LAG2", F.lag("_SR_CURRENT", 2).over(w_sg))
    .withColumn("_SR_LAG3", F.lag("_SR_CURRENT", 3).over(w_sg))
    .withColumn(
        "AVG_SURVIVAL_RATE_3YR",
        (
            F.coalesce(F.col("COHORT_SURVIVAL_RATE"), F.lit(0)) +   # lag 1
            F.coalesce(F.col("_SR_LAG2"),             F.lit(0)) +   # lag 2
            F.coalesce(F.col("_SR_LAG3"),             F.lit(0))     # lag 3
        ) / F.greatest(
            F.when(F.col("COHORT_SURVIVAL_RATE").isNotNull(), 1).otherwise(0) +
            F.when(F.col("_SR_LAG2").isNotNull(),             1).otherwise(0) +
            F.when(F.col("_SR_LAG3").isNotNull(),             1).otherwise(0),
            F.lit(1)
        )
    )
    .drop("_SR_CURRENT", "_SR_LAG2", "_SR_LAG3")
)

print("Cohort Survival Rate features built (leak-free).")
print("  COHORT_SURVIVAL_RATE   = lag(ENROLLMENT/FEEDER_GRADE, 1)  [cross-grade, lagged]")
print("  AVG_SURVIVAL_RATE_3YR  = avg of lags 1, 2, 3             [no current-year term]")


StatementMeta(, d19cdaca-6fa0-468f-918e-5ca30b324312, 11, Finished, Available, Finished, False)

Cohort Survival Rate features built (leak-free).
  COHORT_SURVIVAL_RATE   = lag(ENROLLMENT/FEEDER_GRADE, 1)  [cross-grade, lagged]
  AVG_SURVIVAL_RATE_3YR  = avg of lags 1, 2, 3             [no current-year term]


---
## Section 5 — School Total & District-Grade Trend Features

### `SCHOOL_TOTAL_LAST_YEAR` 

Sum enrollment per school per year, then **lag by 1 year** before joining.
The model now only sees last year's total — never the current year.

### `DISTRICT_GRADE_ENROLLMENT_LAST_YEAR`
This was already correct in the original code (uses a window lag on the district
total). Retained unchanged.

### `IS_MIGRANT_ANOMALY_YEAR`
SY 2022–2024 saw a large surge of migrant students from Venezuela and Central
America, followed by a sharp drop in 2025-26 as federal immigration enforcement
increased. These years are flagged so the model can down-weight them in training.


In [7]:
# ── 5a. School total enrollment — LAGGED (BUG FE-3 fix) ────────────────────
school_totals_raw = (
    df_agg
    .groupBy("SCHOOL_KEY", TIME_COL)
    .agg(F.sum(LABEL_COL).alias("_school_total"))
)
w_school = Window.partitionBy("SCHOOL_KEY").orderBy(TIME_COL)

school_totals = (
    school_totals_raw
    .withColumn(
        "SCHOOL_TOTAL_LAST_YEAR",          # ← lagged by 1 year (no leakage)
        F.lag("_school_total", 1).over(w_school)
    )
    .select("SCHOOL_KEY", TIME_COL, "SCHOOL_TOTAL_LAST_YEAR")
)
# Note: column is now SCHOOL_TOTAL_LAST_YEAR (not SCHOOL_TOTAL_ENROLLMENT)
# Update the FEATURE_COLS list in training accordingly.
df_feat = df_feat.join(school_totals, on=["SCHOOL_KEY", TIME_COL], how="left")

# ── 5b. District-grade enrollment last year (already correct — retained) ─
district_grade_totals = (
    df_agg
    .groupBy("GRADE", TIME_COL)
    .agg(F.sum(LABEL_COL).alias("_dg_total"))
)
w_dg = Window.partitionBy("GRADE").orderBy(TIME_COL)
district_grade_totals = (
    district_grade_totals
    .withColumn(
        "DISTRICT_GRADE_ENROLLMENT_LAST_YEAR",
        F.lag("_dg_total", 1).over(w_dg)
    )
    .drop("_dg_total")
)
df_feat = df_feat.join(district_grade_totals, on=["GRADE", TIME_COL], how="left")

# ── 5c. Migrant anomaly flag ─────────────────────────────────────────────
df_feat = df_feat.withColumn(
    "IS_MIGRANT_ANOMALY_YEAR",
    F.when(F.col(TIME_COL).between(2022, 2024), F.lit(1)).otherwise(F.lit(0))
)

print("School total (lagged), district-grade, and migrant anomaly flag built.")


StatementMeta(, d19cdaca-6fa0-468f-918e-5ca30b324312, 12, Finished, Available, Finished, False)

School total (lagged), district-grade, and migrant anomaly flag built.


---
## Section 6 — School-Type Features from `dim_school`

### Deduplication fix
`dim_school` can contain multiple rows per `SCHOOL_KEY` (historical records,
soft-deletes, etc.). Joining without deduplication **fans out** every enrollment
row, silently doubling (or more) the feature table size.

**Fix:** Keep the single most-recently-updated row per `SCHOOL_KEY` using a
row-number window ordered by `LAST_UPDATED_TS` descending before joining.

### School-type features
Per the **domain guide** (Section 4), the four school types have fundamentally
different enrollment dynamics:

| Type | Dynamics |
|------|----------|
| Neighborhood (District) | Address-assigned, predictable from local demographics |
| Selective | Demand-constrained, very stable once enrolled |
| Magnet | Lottery-based, more random year-to-year |
| Charter | Privately managed, can diverge sharply from district trends |


In [8]:
dim = spark.read.table(DIM_TABLE)
print(f"dim_school raw rows       : {dim.count():,}")
print(f"dim_school distinct keys  : {dim.select('SCHOOL_KEY').distinct().count():,}")

# ── BUG FE-4 FIX: deduplicate to 1 row per SCHOOL_KEY ───────────────────
w_dim = Window.partitionBy("SCHOOL_KEY").orderBy(F.col("LAST_UPDATED_TS").desc_nulls_last())
dim_dedup = (
    dim
    .withColumn("_rn", F.row_number().over(w_dim))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)
print(f"dim_school after dedupe   : {dim_dedup.count():,} (1 row per SCHOOL_KEY)")

# Detect a NETWORK-like column and normalize it to a unified 'NETWORK' field
network_candidates = [c for c in dim_dedup.columns if "NETWORK" in c.upper()]
NETWORK_COL_DIM = network_candidates[0] if network_candidates else None
if NETWORK_COL_DIM is not None:
    print(f"Using NETWORK column from dim_school: '{NETWORK_COL_DIM}'")
else:
    print("No NETWORK column found in dim_school; 'NETWORK' will be NULL in features.")

base_cols = [
    "SCHOOL_KEY", "GOVERNANCE", "SCHOOL_STATUS", "SCHOOL_SUBTYPE",
    "SCHOOL_GRADES_GROUP", "SELECTIVE_ENROLLMENT_TYPE",
    "ATTENDANCE_BOUNDARY", "ANNUAL_REGIONAL_ANALYSIS_REGION", "COMMUNITY",
]

if NETWORK_COL_DIM is not None:
    dim_selected = dim_dedup.select(*base_cols, F.col(NETWORK_COL_DIM).alias("NETWORK"))
else:
    dim_selected = dim_dedup.select(*base_cols).withColumn("NETWORK", F.lit(None).cast("string"))


dim_features = (
    dim_selected
    .withColumn(
        "GOVERNANCE_ENCODED",
        F.when(F.upper(F.col("GOVERNANCE")) == "CHARTER",  F.lit(1))
         .when(F.upper(F.col("GOVERNANCE")) == "CONTRACT", F.lit(2))
         .otherwise(F.lit(0))
    )
    .withColumn(
        "IS_SELECTIVE",
        F.when(
            (F.col("SELECTIVE_ENROLLMENT_TYPE").isNotNull()) &
            (F.trim(F.col("SELECTIVE_ENROLLMENT_TYPE")) != "") &
            (F.upper(F.trim(F.col("SELECTIVE_ENROLLMENT_TYPE"))) != "NON"),
            F.lit(1)
        ).otherwise(F.lit(0))
    )
    .withColumn(
        "IS_ATTENDANCE_AREA",
        F.when(F.upper(F.trim(F.col("ATTENDANCE_BOUNDARY"))) == "ATTENDANCE-AREA",
               F.lit(1)).otherwise(F.lit(0))
    )
    .withColumn(
        "IS_SMALL_SCHOOL",
        F.when(F.upper(F.trim(F.col("SCHOOL_SUBTYPE"))) == "SMALL",
               F.lit(1)).otherwise(F.lit(0))
    )
    .withColumn(
        "IS_HIGH_SCHOOL",
        F.when(F.upper(F.trim(F.col("SCHOOL_GRADES_GROUP"))) == "HIGH",
               F.lit(1)).otherwise(F.lit(0))
    )
    .withColumn(
        "IS_SCHOOL_OPEN",
        F.when(F.upper(F.trim(F.col("SCHOOL_STATUS"))) == "OPEN",
               F.lit(1)).otherwise(F.lit(0))
    )
    .withColumn(
        "REGION_ENCODED",
        F.when(F.col("ANNUAL_REGIONAL_ANALYSIS_REGION") == "South Side",                    F.lit(1))
         .when(F.col("ANNUAL_REGIONAL_ANALYSIS_REGION") == "West Side",                     F.lit(2))
         .when(F.col("ANNUAL_REGIONAL_ANALYSIS_REGION") == "North Lakefront",               F.lit(3))
         .when(F.col("ANNUAL_REGIONAL_ANALYSIS_REGION") == "Northwest Side",                F.lit(4))
         .when(F.col("ANNUAL_REGIONAL_ANALYSIS_REGION") == "Far Northwest Side",            F.lit(5))
         .when(F.col("ANNUAL_REGIONAL_ANALYSIS_REGION") == "Bronzeville / South Lakefront", F.lit(6))
         .when(F.col("ANNUAL_REGIONAL_ANALYSIS_REGION") == "Greater Stony Island",          F.lit(7))
         .when(F.col("ANNUAL_REGIONAL_ANALYSIS_REGION") == "Greater Calumet",               F.lit(8))
         .when(F.col("ANNUAL_REGIONAL_ANALYSIS_REGION") == "Greater Midway",                F.lit(9))
         .when(F.col("ANNUAL_REGIONAL_ANALYSIS_REGION") == "Greater Stockyards",            F.lit(10))
         .when(F.col("ANNUAL_REGIONAL_ANALYSIS_REGION") == "Pilsen / Little Village",       F.lit(11))
         .when(F.col("ANNUAL_REGIONAL_ANALYSIS_REGION") == "Near West Side",                F.lit(12))
         .when(F.col("ANNUAL_REGIONAL_ANALYSIS_REGION") == "Greater Milwaukee Avenue",      F.lit(13))
         .when(F.col("ANNUAL_REGIONAL_ANALYSIS_REGION") == "Greater Lincoln Park",          F.lit(14))
         .otherwise(F.lit(0))
    )
    .select(
        "SCHOOL_KEY", "GOVERNANCE_ENCODED", "IS_SELECTIVE", "IS_ATTENDANCE_AREA",
        "IS_SMALL_SCHOOL", "IS_HIGH_SCHOOL", "IS_SCHOOL_OPEN", "REGION_ENCODED",
        "GOVERNANCE", "ATTENDANCE_BOUNDARY", "ANNUAL_REGIONAL_ANALYSIS_REGION", "COMMUNITY",
        "NETWORK",
    )
)


df_feat = df_feat.join(dim_features, on="SCHOOL_KEY", how="left")
df_feat = df_feat.fillna({
    "GOVERNANCE_ENCODED": 0, "IS_SELECTIVE": 0, "IS_ATTENDANCE_AREA": 0,
    "IS_SMALL_SCHOOL": 0, "IS_HIGH_SCHOOL": 0, "IS_SCHOOL_OPEN": 1, "REGION_ENCODED": 0,
})

print("\ndim_school features joined (including NETWORK where available).")

StatementMeta(, d19cdaca-6fa0-468f-918e-5ca30b324312, 13, Finished, Available, Finished, False)

dim_school raw rows       : 5,830


dim_school distinct keys  : 2,915
dim_school after dedupe   : 2,915 (1 row per SCHOOL_KEY)
Using NETWORK column from dim_school: 'NETWORK'

dim_school features joined (including NETWORK where available).


---
## Section 7 — Build & Write Final Feature Table

Selects the target grades (1–8, 10–12; K and Grade 9 are entry grades that need
their own models), retains only rows that have at least one year of lag history
(`SAME_GRADE_LAST_YEAR` is not null), and writes to the Lakehouse.

> **Fill-rate check:** Every model feature is printed with its non-null percentage.
> A `⚠` flag appears on any feature below 80% — investigate before training.


In [9]:
final_columns = [
    "SCHOOL_KEY", "GRADE", "GRADE_NUMERIC", TIME_COL,
    LABEL_COL,
    "SAME_GRADE_LAST_YEAR", "SAME_GRADE_2YR_AGO",
    "FEEDER_GRADE_LAST_YEAR", "FEEDER_GRADE_2YR_AGO", "HAS_FEEDER_GRADE",
    "SCHOOL_TOTAL_LAST_YEAR",            # renamed from SCHOOL_TOTAL_ENROLLMENT
    "COHORT_SURVIVAL_RATE",              # leak-free: lag(cross-grade SR, 1)
    "AVG_SURVIVAL_RATE_3YR",             # leak-free: avg(lag 1,2,3)
    "DISTRICT_GRADE_ENROLLMENT_LAST_YEAR",
    "IS_MIGRANT_ANOMALY_YEAR",
    "GOVERNANCE_ENCODED", "IS_SELECTIVE", "IS_ATTENDANCE_AREA",
    "IS_SMALL_SCHOOL", "IS_HIGH_SCHOOL", "REGION_ENCODED",
    "IS_SCHOOL_OPEN",
    "GOVERNANCE", "ATTENDANCE_BOUNDARY", "ANNUAL_REGIONAL_ANALYSIS_REGION", "COMMUNITY",
    "NETWORK",
]


df_final = (
    df_feat
    .filter(F.col("GRADE").isin(TARGET_GRADES))
    .filter(F.col("SAME_GRADE_LAST_YEAR").isNotNull())
    .select(final_columns)
)

final_count = df_final.count()
print(f"Final feature table: {final_count:,} rows × {len(final_columns)} columns")

# Feature fill-rate check
model_features = [
    "SAME_GRADE_LAST_YEAR", "SAME_GRADE_2YR_AGO",
    "FEEDER_GRADE_LAST_YEAR", "FEEDER_GRADE_2YR_AGO", "HAS_FEEDER_GRADE",
    "SCHOOL_TOTAL_LAST_YEAR", "COHORT_SURVIVAL_RATE", "AVG_SURVIVAL_RATE_3YR",
    "DISTRICT_GRADE_ENROLLMENT_LAST_YEAR", "IS_MIGRANT_ANOMALY_YEAR",
    "GOVERNANCE_ENCODED", "IS_SELECTIVE", "IS_ATTENDANCE_AREA",
    "IS_SMALL_SCHOOL", "IS_HIGH_SCHOOL", "REGION_ENCODED",
]
print("\nFeature fill rates:")
for col_name in model_features:
    non_null = df_final.filter(F.col(col_name).isNotNull()).count()
    pct = round(non_null / final_count * 100, 1)
    flag = "✔" if pct >= 80 else "⚠"
    print(f"  {flag}  {col_name:45s}: {pct}%")


df_final.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(FEATURE_TABLE)
print(f"\n✔ Written: '{FEATURE_TABLE}'")
print(f"  Rows   : {final_count:,}  |  Columns: {len(final_columns)}")


StatementMeta(, d19cdaca-6fa0-468f-918e-5ca30b324312, 14, Finished, Available, Finished, False)

Final feature table: 36,534 rows × 27 columns

Feature fill rates:


  ✔  SAME_GRADE_LAST_YEAR                         : 100.0%


  ✔  SAME_GRADE_2YR_AGO                           : 83.6%


  ✔  FEEDER_GRADE_LAST_YEAR                       : 100.0%


...

  ✔  IS_ATTENDANCE_AREA                           : 100.0%


  ✔  IS_SMALL_SCHOOL                              : 100.0%


  ✔  IS_HIGH_SCHOOL                               : 100.0%


  ✔  REGION_ENCODED                               : 100.0%



✔ Written: 'ml_enrollment_feature_table_v4'
  Rows   : 36,534  |  Columns: 27


---
## Section 8 — Load Feature Table & Time-Based Split

Reload the saved feature table (or continue with `df_final` in memory), then
split strictly by year.

> **Critical:** **Never split by random sample on a time-series problem.**
> Splitting randomly would let the model train on 2025 data to predict 2022 —
> the future leaks into the past. Always train on years ≤ T−1 and validate on year T.


In [10]:
df = spark.read.table(FEATURE_TABLE)

year_stats = df.select(F.min(TIME_COL).alias("min_y"), F.max(TIME_COL).alias("max_y")).collect()[0]
print(f"Feature table year range: {year_stats['min_y']} – {year_stats['max_y']}")
print(f"Configured  train window: {TRAIN_START_YEAR} – {TRAIN_END_YEAR}")
print(f"Configured    val window: {VAL_START_YEAR}   – {VAL_END_YEAR}")

train_df = df.filter((F.col(TIME_COL) >= TRAIN_START_YEAR) & (F.col(TIME_COL) <= TRAIN_END_YEAR))
val_df   = df.filter((F.col(TIME_COL) >= VAL_START_YEAR)   & (F.col(TIME_COL) <= VAL_END_YEAR))

print(f"\nTrain rows : {train_df.count():,}")
print(f"Val rows   : {val_df.count():,}")


StatementMeta(, d19cdaca-6fa0-468f-918e-5ca30b324312, 15, Finished, Available, Finished, False)

Feature table year range: 2020 – 2026
Configured  train window: 2020 – 2025
Configured    val window: 2026   – 2026

Train rows : 31,005
Val rows   : 5,529


## Section 11b — Grade Coverage & Entry-Grade (K, 9) Audit

`TARGET_GRADES` *requests* all of K–12, but a grade only survives into the feature
table if the **source data actually carries it** and it clears the `GRADE_ORDER`
map. This audit shows which grades made it into training, how many rows each has,
and how often a grade has a **real feeder** (`HAS_FEEDER_GRADE = 1`). Entry grades
(K, and 9 at the elementary→high-school break) have **no feeder grade inside the
school**, so they are expected to read ~0% real-feeder and lean on imputed lags.

In [ ]:
# ── Are K and Grade 9 actually in training? Where do they drop out? ──────────
print("TARGET_GRADES (requested):", TARGET_GRADES)

present = sorted(
    [r["GRADE"] for r in df.select("GRADE").distinct().collect()],
    key=lambda g: GRADE_ORDER.get(g, 99),
)
print("Grades present in feature table:", present)

missing = [g for g in TARGET_GRADES if g not in present]
print("Requested grades MISSING from the data:", missing or "none")

# Per-grade training rows + share with a REAL feeder grade (entry grades -> ~0%)
print("\nPer-grade training coverage (years <= TRAIN_END_YEAR):")
grade_audit = (
    train_df.groupBy("GRADE")
    .agg(
        F.count("*").alias("rows"),
        F.round(F.avg("HAS_FEEDER_GRADE") * 100, 1).alias("pct_real_feeder"),
    )
    .collect()
)
for r in sorted(grade_audit, key=lambda r: GRADE_ORDER.get(r["GRADE"], 99)):
    note = "   <- ENTRY GRADE (no internal feeder)" if r["pct_real_feeder"] == 0 else ""
    print(f"  Grade {str(r['GRADE']):>2}: {r['rows']:>7,} rows | "
          f"{r['pct_real_feeder']:>5}% have a real feeder{note}")

# Smoking gun: what grade codes does the RAW source carry, before mapping?
try:
    raw_grades = sorted(
        str(x[0]) for x in df_raw.select("ENTRY_GRADE_LEVEL").distinct().collect()
        if x[0] is not None
    )
    print("\nRaw ENTRY_GRADE_LEVEL codes in source:", raw_grades)
    unmapped = [g for g in raw_grades if g not in GRADE_ORDER]
    print("Raw codes NOT in GRADE_ORDER (silently dropped at the map):", unmapped or "none")
    print("  -> If Kindergarten/Grade 9 appear here as e.g. 'KG'/'09', add those keys")
    print("     to GRADE_ORDER so they survive into training.")
except Exception as e:
    print(f"\n(Raw grade-code check skipped - df_raw not in this session: {e})")

---
## Section 9 — NULL Imputation

Medians are computed **from training data only**. Applying training-set medians to
the validation set prevents data leakage from the validation fold into imputation.

Columns filled here are those that are legitimately null for schools in their
first few years of operation (no lag history yet).


In [11]:
FEATURE_COLS = [
    "SAME_GRADE_LAST_YEAR",
    "SAME_GRADE_2YR_AGO",
    "FEEDER_GRADE_LAST_YEAR",
    "FEEDER_GRADE_2YR_AGO",
    "HAS_FEEDER_GRADE",
    "SCHOOL_TOTAL_LAST_YEAR",            # renamed from SCHOOL_TOTAL_ENROLLMENT
    "COHORT_SURVIVAL_RATE",
    "AVG_SURVIVAL_RATE_3YR",
    "DISTRICT_GRADE_ENROLLMENT_LAST_YEAR",
    "IS_MIGRANT_ANOMALY_YEAR",
    "GRADE_NUMERIC",
    "SCHOOL_KEY",
    "GOVERNANCE_ENCODED",
    "IS_SELECTIVE",
    "IS_ATTENDANCE_AREA",
    "IS_SMALL_SCHOOL",
    "IS_HIGH_SCHOOL",
    "REGION_ENCODED",
]

null_fill_cols = [
    "SAME_GRADE_2YR_AGO",
    "FEEDER_GRADE_LAST_YEAR",
    "FEEDER_GRADE_2YR_AGO",
    "COHORT_SURVIVAL_RATE",
    "AVG_SURVIVAL_RATE_3YR",
    "DISTRICT_GRADE_ENROLLMENT_LAST_YEAR",
    "SCHOOL_TOTAL_LAST_YEAR",
]

print("Computing medians from TRAINING data only:")
median_fills = {}
for col_name in null_fill_cols:
    q = train_df.filter(F.col(col_name).isNotNull()).approxQuantile(col_name, [0.5], 0.01)
    median_fills[col_name] = q[0] if q else 0.0
    print(f"  {col_name:45s}: {median_fills[col_name]:.3f}")

train_df = train_df.fillna(median_fills)
val_df   = val_df.fillna(median_fills)
print("\nMedian imputation applied to train and val.")


StatementMeta(, d19cdaca-6fa0-468f-918e-5ca30b324312, 16, Finished, Available, Finished, False)

Computing medians from TRAINING data only:


  SAME_GRADE_2YR_AGO                           : 26.000


  FEEDER_GRADE_LAST_YEAR                       : 30.000


  FEEDER_GRADE_2YR_AGO                         : 31.000
  COHORT_SURVIVAL_RATE                         : 0.786


  AVG_SURVIVAL_RATE_3YR                        : 0.662
  DISTRICT_GRADE_ENROLLMENT_LAST_YEAR          : 14666.000
  SCHOOL_TOTAL_LAST_YEAR                       : 454.000

Median imputation applied to train and val.


---
## Section 10 — Sample Weights

School years 2022–2024 contain anomalous enrollment spikes caused by the
Chicago migrant arrival surge (Venezuelan / Central American families).
These years are **not excluded** — the pattern is real — but they are
**down-weighted to 0.3** so they influence training less than normal years.

Validation rows always get weight 1.0 (evaluation should reflect actual conditions).


In [12]:
train_df = train_df.withColumn(
    WEIGHT_COL,
    F.when(F.col("IS_MIGRANT_ANOMALY_YEAR") == 1, F.lit(0.3)).otherwise(F.lit(1.0))
)
val_df = val_df.withColumn(WEIGHT_COL, F.lit(1.0))

migrant_rows = train_df.filter(F.col("IS_MIGRANT_ANOMALY_YEAR") == 1).count()
normal_rows  = train_df.filter(F.col("IS_MIGRANT_ANOMALY_YEAR") == 0).count()
print(f"Sample weights — normal: {normal_rows:,} (w=1.0)  |  anomaly: {migrant_rows:,} (w=0.3)")


StatementMeta(, d19cdaca-6fa0-468f-918e-5ca30b324312, 17, Finished, Available, Finished, False)

Sample weights — normal: 14,681 (w=1.0)  |  anomaly: 16,324 (w=0.3)


---
## Section 11 — NaN/Inf Cleaning & Model Training

### Model
GBT Regressor with fixed hyperparameters, operating on a `Pipeline` of:
`StringIndexer(GRADE → GRADE_idx)` → `VectorAssembler` → `GBTRegressor`


In [13]:
def compute_metrics(pred_df, label_col, pred_col):
    """Compute RMSE, MAE, R² using Spark evaluators."""
    evs = {
        "RMSE": RegressionEvaluator(labelCol=label_col, predictionCol=pred_col, metricName="rmse"),
        "MAE":  RegressionEvaluator(labelCol=label_col, predictionCol=pred_col, metricName="mae"),
        "R2":   RegressionEvaluator(labelCol=label_col, predictionCol=pred_col, metricName="r2"),
    }
    return {name: ev.evaluate(pred_df) for name, ev in evs.items()}


# ── BUG MT-2 FIX: type-aware NaN / Inf check ────────────────────────────
float_type_cols = {
    f.name for f in train_df.schema.fields
    if str(f.dataType) in ("DoubleType()", "FloatType()")
}

print("Checking training data for NaN / Inf in feature columns...")
for c in FEATURE_COLS:
    if c in float_type_cols:
        bad_cnt = train_df.filter(
            F.col(c).isNull() | F.isnan(c) |
            (F.col(c) == float("inf")) | (F.col(c) == float("-inf"))
        ).count()
    else:
        bad_cnt = train_df.filter(F.col(c).isNull()).count()
    if bad_cnt > 0:
        print(f"  {c}: {bad_cnt} problematic rows — will be dropped")

def clean_df(sdf, float_cols, all_feature_cols):
    """Drop rows with NaN/Inf (floats) or NULL (ints) in any feature column."""
    out = sdf
    for c in all_feature_cols:
        if c in float_cols:
            out = out.filter(
                ~F.isnan(c) &
                (F.col(c) != float("inf")) &
                (F.col(c) != float("-inf")) &
                F.col(c).isNotNull()
            )
        else:
            out = out.filter(F.col(c).isNotNull())
    return out

train_df_clean = clean_df(train_df, float_type_cols, FEATURE_COLS)
# BUG MT-3 FIX: clean val_df too — was missing in original code
val_df_clean   = clean_df(val_df,   float_type_cols, FEATURE_COLS)

print(f"Train rows before cleaning : {train_df.count():,}")
print(f"Train rows after  cleaning : {train_df_clean.count():,}")
print(f"Val   rows after  cleaning : {val_df_clean.count():,}")

# ── Build pipeline ───────────────────────────────────────────────────────
grade_indexer = StringIndexer(inputCol="GRADE", outputCol="GRADE_idx", handleInvalid="keep")
all_feature_cols = FEATURE_COLS + ["GRADE_idx"]
assembler = VectorAssembler(inputCols=all_feature_cols, outputCol="features", handleInvalid="keep")

gbt = GBTRegressor(
    labelCol=LABEL_COL, featuresCol="features", weightCol=WEIGHT_COL,
    seed=42, maxBins=256,
    maxDepth=BEST_MAX_DEPTH, maxIter=BEST_MAX_ITER,
    stepSize=BEST_STEP_SIZE, subsamplingRate=BEST_SUBSAMPLING_RATE,
)

pipeline = Pipeline(stages=[grade_indexer, assembler, gbt])

print(f"\nTraining GBT (maxDepth={BEST_MAX_DEPTH}, maxIter={BEST_MAX_ITER}, "
      f"stepSize={BEST_STEP_SIZE}, subsampling={BEST_SUBSAMPLING_RATE}) ...")
model = pipeline.fit(train_df_clean)
print("Training complete.")

val_pred = model.transform(val_df_clean)
val_pred = (
    val_pred
    .withColumn("prediction", F.col("prediction").cast(DoubleType()))
    .withColumn(LABEL_COL,    F.col(LABEL_COL).cast(DoubleType()))
)

gbt_metrics = compute_metrics(val_pred, LABEL_COL, "prediction")
best_metrics = gbt_metrics

print(f"\n{'='*60}")
print(f"  GBT MODEL — Validation Metrics (SY {VAL_START_YEAR}–{VAL_END_YEAR})")
print(f"{'='*60}")
print(f"  RMSE : {gbt_metrics['RMSE']:.2f}")
print(f"  MAE  : {gbt_metrics['MAE']:.2f}")
print(f"{'='*60}")


StatementMeta(, d19cdaca-6fa0-468f-918e-5ca30b324312, 18, Finished, Available, Finished, False)

Checking training data for NaN / Inf in feature columns...


Train rows before cleaning : 31,005
Train rows after  cleaning : 31,005
Val   rows after  cleaning : 5,529



Training GBT (maxDepth=5, maxIter=100, stepSize=0.1, subsampling=0.8) ...


Training complete.



  GBT MODEL — Validation Metrics (SY 2026–2026)
  RMSE : 43.10
  MAE  : 10.70


In [14]:
import os
import shutil

# Save the fully trained PipelineModel (with GBTRegressor) as a ZIP file
# Assumes `model` is the trained PipelineModel from Section 11.

# 1. Local temp folder on the driver to save the Spark model
local_model_dir = "/tmp/gbt_enrollment_model"
local_model_uri = f"file:{local_model_dir}"  # Spark expects a URI

# Clean up any previous run
if os.path.exists(local_model_dir):
    shutil.rmtree(local_model_dir)

# 2. Save the full PipelineModel (includes GBTRegressor) locally
model.write().overwrite().save(local_model_uri)
print(f"Model saved to local directory: {local_model_dir}")

# 3. Zip the saved model folder
zip_base = "/tmp/gbt_enrollment_model"          # base path (without .zip)
zip_path = shutil.make_archive(zip_base, "zip", local_model_dir)
print(f"Created ZIP archive at: {zip_path}")

# 4. Copy the ZIP into Lakehouse Files in new directory 'models'
target_dir_in_lakehouse = "Files/models"
notebookutils.fs.mkdirs(target_dir_in_lakehouse)

target_in_lakehouse = f"{target_dir_in_lakehouse}/gbt_enrollment_model.zip"
notebookutils.fs.cp(f"file:{zip_path}", target_in_lakehouse, recurse=False)

print(f"Saved model ZIP to Lakehouse: {target_in_lakehouse}")

StatementMeta(, d19cdaca-6fa0-468f-918e-5ca30b324312, 19, Finished, Available, Finished, False)

Model saved to local directory: /tmp/gbt_enrollment_model
Created ZIP archive at: /tmp/gbt_enrollment_model.zip
Saved model ZIP to Lakehouse: Files/models/gbt_enrollment_model.zip


---
## Section 12 — CSR Baseline Comparison

The correct formula is:
```python
# ✅ Fixed — matches domain definition exactly
csr_prediction = FEEDER_GRADE_LAST_YEAR * AVG_SURVIVAL_RATE_3YR
```
where `AVG_SURVIVAL_RATE_3YR` is the leak-free 3-year average cross-grade ratio
built in Section 4. This is exactly what the current SAS system does.


In [15]:
# CSR baseline = FEEDER_GRADE_LAST_YEAR * AVG_SURVIVAL_RATE_3YR
# Both columns are leak-free (fully lagged) from Section 4 and 5.
# This matches the domain guide definition exactly.
val_pred_csr = (
    val_pred
    .withColumn(
        "csr_prediction",
        (
            F.col("FEEDER_GRADE_LAST_YEAR") *
            F.coalesce(F.col("AVG_SURVIVAL_RATE_3YR"), F.lit(1.0))
        ).cast(DoubleType())
    )
)

csr_metrics = compute_metrics(val_pred_csr, LABEL_COL, "csr_prediction")

print(f"\n{'='*60}")
print(f"  CSR BASELINE  (FEEDER_GRADE_LAST_YEAR × AVG_SURVIVAL_RATE_3YR)")
print(f"  [Domain guide definition: grade N-1 count × historical cross-grade ratio]")
print(f"{'='*60}")
print(f"  RMSE : {csr_metrics['RMSE']:.2f}")
print(f"  MAE  : {csr_metrics['MAE']:.2f}")

rmse_improvement = (csr_metrics['RMSE'] - gbt_metrics['RMSE']) / csr_metrics['RMSE'] * 100
print(f"\n  ML improvement over CSR baseline: {rmse_improvement:.1f}% RMSE reduction")
if rmse_improvement > 0:
    print("  ✔ ML model is BETTER than the CSR baseline.")
else:
    print("  ✘ ML model is NOT better than CSR yet — review features or hyperparameters.")
print(f"{'='*60}")


StatementMeta(, d19cdaca-6fa0-468f-918e-5ca30b324312, 20, Finished, Available, Finished, False)


  CSR BASELINE  (FEEDER_GRADE_LAST_YEAR × AVG_SURVIVAL_RATE_3YR)
  [Domain guide definition: grade N-1 count × historical cross-grade ratio]
  RMSE : 4358.18
  MAE  : 249.83

  ML improvement over CSR baseline: 99.0% RMSE reduction
  ✔ ML model is BETTER than the CSR baseline.


---
## Section 13 — Feature Importance

GBT feature importances show how much each feature contributed to split decisions
across all trees. High importance for `COHORT_SURVIVAL_RATE` after the leakage fix
indicates genuine predictive signal — not the model reading the answer off the label.


In [16]:
gbt_model   = model.stages[-1]
importances = gbt_model.featureImportances.toArray()

fi_df = pd.DataFrame({
    "feature":    all_feature_cols,
    "importance": importances,
}).sort_values("importance", ascending=False).reset_index(drop=True)

print("Feature Importance:")
for _, row in fi_df.iterrows():
    bar = "█" * int(row["importance"] * 40)
    print(f"  {row['feature']:45s} {row['importance']:.4f}  {bar}")

v3_features = ["GOVERNANCE_ENCODED", "IS_SELECTIVE", "IS_ATTENDANCE_AREA",
               "IS_SMALL_SCHOOL", "IS_HIGH_SCHOOL", "REGION_ENCODED"]
v3_importance = fi_df[fi_df["feature"].isin(v3_features)]["importance"].sum()
print(f"\n  V3 school-type features combined importance: {v3_importance:.4f}")
print(f"  (> 0.05 indicates school type is meaningfully contributing)")


StatementMeta(, d19cdaca-6fa0-468f-918e-5ca30b324312, 21, Finished, Available, Finished, False)

Feature Importance:
  SAME_GRADE_LAST_YEAR                          0.5825  ███████████████████████
  SCHOOL_TOTAL_LAST_YEAR                        0.0955  ███
  DISTRICT_GRADE_ENROLLMENT_LAST_YEAR           0.0541  ██
  FEEDER_GRADE_2YR_AGO                          0.0422  █
  GRADE_idx                                     0.0398  █
  SCHOOL_KEY                                    0.0394  █
  REGION_ENCODED                                0.0360  █
  COHORT_SURVIVAL_RATE                          0.0237  
  FEEDER_GRADE_LAST_YEAR                        0.0223  
  SAME_GRADE_2YR_AGO                            0.0189  
  AVG_SURVIVAL_RATE_3YR                         0.0125  
  IS_ATTENDANCE_AREA                            0.0088  
  IS_HIGH_SCHOOL                                0.0086  
  IS_SELECTIVE                                  0.0065  
  HAS_FEEDER_GRADE                              0.0046  
  IS_MIGRANT_ANOMALY_YEAR                       0.0025  
  GRADE_NUMERIC                     

---
## Section 14 — Forecast Next School Year

Generates enrollment predictions for `FORECAST_END_YEAR` using the trained model.
Only **open schools** are included — forecasting closed schools is meaningless.

The latest available row per school-grade is used as the feature template.
`SCHOOL_YEAR` is bumped to the forecast year and `IS_MIGRANT_ANOMALY_YEAR` is set
to 0 (the anomaly window is treated as over for future forecasts).


In [17]:
forecast_year = FORECAST_END_YEAR

w_latest = Window.partitionBy("SCHOOL_KEY", "GRADE").orderBy(F.col(TIME_COL).desc())

latest_per_group = (
    df
    .filter(F.col("IS_SCHOOL_OPEN") == 1)
    .fillna(median_fills)
    .withColumn("_rn", F.row_number().over(w_latest))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
    .withColumn(TIME_COL, F.lit(forecast_year))
    .withColumn(WEIGHT_COL, F.lit(1.0))
    .withColumn("IS_MIGRANT_ANOMALY_YEAR", F.lit(0))
)

forecast_pred = model.transform(latest_per_group)

forecast_out = (
    forecast_pred
    .select(
        "SCHOOL_KEY", "GOVERNANCE", "ANNUAL_REGIONAL_ANALYSIS_REGION", "COMMUNITY", "GRADE",
        F.col(TIME_COL).alias("FORECAST_YEAR"),
        F.round("prediction", 0).alias("PREDICTED_ENROLLMENT"),
        "IS_SELECTIVE", "IS_ATTENDANCE_AREA",
    )
    .orderBy("SCHOOL_KEY", "GRADE")
)

print(f"Forecast for SY{forecast_year} (open schools only):")
print(f"  School-grade combinations: {forecast_out.count():,}")
display(forecast_out.limit(30))

FORECAST_TABLE = f"enrollment_forecast_sy{forecast_year}_{FORECAST_TABLE_SUFFIX}"
forecast_out.write.mode("overwrite").saveAsTable(FORECAST_TABLE)
print(f"\n✔ Forecast saved to: '{FORECAST_TABLE}'")


StatementMeta(, d19cdaca-6fa0-468f-918e-5ca30b324312, 22, Finished, Available, Finished, False)

Forecast for SY2027 (open schools only):


  School-grade combinations: 5,911


SynapseWidget(Synapse.DataFrame, 08a2454f-55f5-4cf4-8775-64dccdd5e720)


✔ Forecast saved to: 'enrollment_forecast_sy2027_v3_fixed'


---
## Section 15 — Pipeline Summary

| Stage | Table | Description |
|-------|-------|-------------|
| Feature Engineering | `ml_enrollment_feature_table_v3` | 1 row per school × grade × year, all leak-free features |
| Model Training | in-memory `model` | GBT pipeline, fixed hyperparameters |
| Validation Metrics | printed above | RMSE / MAE / R² vs CSR baseline |
| Forecast | `enrollment_forecast_sy{YEAR}_v3_fixed` | Predicted enrollment for open schools only |

### Next Steps
- **Grade 9 separate model** — entry grade; needs GoCPS application data (offers/acceptances from prior year)
- **Kindergarten separate model** — entry grade; needs ILDPH birth data (5-year lag)
- **Hyperparameter tuning** — use the commented-out grid-search cell (Section 6 of the original training notebook) once the leak-free baseline is established
- **Student-level features** — add `%EL`, `%SPED`, `%FRM` per school-grade-year from `dim_student_annual_attrib` as lagged features


---
# PART B — CSR vs ML Model Comparison

Everything below builds directly on the pipeline above (it reuses the in-session
`df` feature table, the `pipeline` object, `clean_df`, `FEATURE_COLS`,
`null_fill_cols`, `float_type_cols`, and the year/label/weight config).

**Aggregation levels used (mapped to the columns that actually exist in CPS data):**

| Prompt level | This data |
|--------------|-----------|
| School       | `SCHOOL_KEY` |
| District     | CPS-wide overall (CPS *is* one district) |
| Network      | `NETWORK` column from `dim_school` |

## Section 16 — Walk-Forward Backtest Setup

The notebook validates on a single year (2026). A *year-wise* CSR-vs-ML
comparison needs out-of-sample predictions for **several** years, so we run an
**expanding-window walk-forward**:

> For each evaluation year `T`: train ML on years `[TRAIN_START_YEAR, T-1]`,
> predict year `T`. CSR is leak-free in every year by construction.

This makes both prediction tables genuinely out-of-sample and the year-wise
metrics fair. The 2026 fold should reproduce the headline metrics from Section 11
(RMSE ≈ 35.8, MAE ≈ 8.1) — a built-in sanity check.

We also attach the `NETWORK` column (auto-detected from `dim_school`) here.


In [18]:
import matplotlib.pyplot as plt

# ── 16a. Attach / normalize NETWORK on the feature table ───────────────────
network_candidates = [c for c in dim.columns if "NETWORK" in c.upper()]
print(f"Candidate network columns in dim_school: {network_candidates}")
NETWORK_COL = network_candidates[0] if network_candidates else None

if "NETWORK" in df.columns:
    # NETWORK already present from feature engineering; just clean it up.
    df_net = df.withColumn(
        "NETWORK",
        F.coalesce(F.trim(F.col("NETWORK").cast("string")), F.lit("UNKNOWN"))
    )
    print("Using NETWORK column already present in feature table.")
elif NETWORK_COL is not None:
    # Fallback: join from dim_school if not already in df
    net_lookup = dim_dedup.select("SCHOOL_KEY", F.col(NETWORK_COL).alias("NETWORK"))
    df_net = df.join(net_lookup, on="SCHOOL_KEY", how="left")
    df_net = df_net.withColumn(
        "NETWORK",
        F.coalesce(F.trim(F.col("NETWORK").cast("string")), F.lit("UNKNOWN"))
    )
    print(f"Using NETWORK column from dim_school: '{NETWORK_COL}'")
else:
    # Last-resort fallback so the rest of the notebook still runs end-to-end.
    df_net = df.withColumn("NETWORK", F.coalesce(F.col("GOVERNANCE"), F.lit("UNKNOWN")))
    print("No NETWORK column found -> falling back to GOVERNANCE as 'NETWORK'.")

print(f"Distinct networks: {df_net.select('NETWORK').distinct().count()}")


# ── 16b. Per-fold helpers (mirror the notebook's train-only imputation) ───
def impute_train_only(train_sdf, test_sdf, cols):
    """Compute medians from TRAIN only; apply to both train and test (no leakage)."""
    fills = {}
    for c in cols:
        q = train_sdf.filter(F.col(c).isNotNull()).approxQuantile(c, [0.5], 0.01)
        fills[c] = q[0] if q else 0.0
    return train_sdf.fillna(fills), test_sdf.fillna(fills)


def add_weights(train_sdf, test_sdf):
    """Down-weight migrant-anomaly years (2022-24) in training; test weight = 1.0."""
    train_sdf = train_sdf.withColumn(
        WEIGHT_COL,
        F.when(F.col("IS_MIGRANT_ANOMALY_YEAR") == 1, F.lit(0.3)).otherwise(F.lit(1.0))
    )
    test_sdf = test_sdf.withColumn(WEIGHT_COL, F.lit(1.0))
    return train_sdf, test_sdf


def csr_prediction_col(sdf):
    """Faithful CSR baseline: FEEDER_GRADE_LAST_YEAR * AVG_SURVIVAL_RATE_3YR."""
    return sdf.withColumn(
        "y_hat_CSR",
        (F.col("FEEDER_GRADE_LAST_YEAR") *
         F.coalesce(F.col("AVG_SURVIVAL_RATE_3YR"), F.lit(1.0))).cast(DoubleType())
    )


# Evaluation years: need >= a few years of history before the first fold.
BACKTEST_START_YEAR = TRAIN_START_YEAR + 3          # 2023 by default
data_max_year = df_net.select(F.max(TIME_COL)).collect()[0][0]
EVAL_YEARS = list(range(BACKTEST_START_YEAR, int(data_max_year) + 1))

print(f"\nExpanding-window backtest")
print(f"  Train start (fixed) : {TRAIN_START_YEAR}")
print(f"  Evaluation years    : {EVAL_YEARS}")


StatementMeta(, d19cdaca-6fa0-468f-918e-5ca30b324312, 23, Finished, Available, Finished, False)

Candidate network columns in dim_school: ['NETWORK', 'NETWORK_CHIEF', 'GEO_NETWORK', 'ES_NETWORK']
Using NETWORK column already present in feature table.


Distinct networks: 23



Expanding-window backtest
  Train start (fixed) : 2020
  Evaluation years    : [2023, 2024, 2025, 2026]


---
## Section 17 — Generate CSR & ML Predictions (Walk-Forward)

For each evaluation year we refit the **same** GBT pipeline on the expanding
training window and predict the held-out year. CSR predictions are computed on
the identical held-out rows. The union of all folds is the out-of-sample
prediction set that powers every metric and chart below.


In [19]:
ID_COLS = ["SCHOOL_KEY", "GRADE", "NETWORK",
           "ANNUAL_REGIONAL_ANALYSIS_REGION", "COMMUNITY", "GOVERNANCE"]

fold_predictions = []

for T in EVAL_YEARS:
    train = df_net.filter((F.col(TIME_COL) >= TRAIN_START_YEAR) & (F.col(TIME_COL) < T))
    test  = df_net.filter(F.col(TIME_COL) == T)

    train, test = impute_train_only(train, test, null_fill_cols)
    train, test = add_weights(train, test)

    train = clean_df(train, float_type_cols, FEATURE_COLS)
    test  = clean_df(test,  float_type_cols, FEATURE_COLS)

    n_train, n_test = train.count(), test.count()
    fold_model = pipeline.fit(train)          # fresh fit on years < T
    pred = fold_model.transform(test)
    pred = csr_prediction_col(pred)

    pred_out = pred.select(
        F.col(TIME_COL).alias("YEAR"),
        *ID_COLS,
        F.col(LABEL_COL).cast(DoubleType()).alias("y"),
        F.col("prediction").cast(DoubleType()).alias("y_hat_ML"),
        F.col("y_hat_CSR"),
    )
    fold_predictions.append(pred_out)
    print(f"  Year {T}: trained on {n_train:,} rows  ->  predicted {n_test:,} rows")

# Union all folds into the unified out-of-sample prediction set
preds_all = fold_predictions[0]
for p in fold_predictions[1:]:
    preds_all = preds_all.unionByName(p)

preds_all = preds_all.cache()
print(f"\nTotal out-of-sample predictions: {preds_all.count():,} rows "
      f"across years {EVAL_YEARS[0]}-{EVAL_YEARS[-1]}")

# Persist for reuse / downstream dashboards
preds_all.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("csr_vs_ml_predictions")
print("Saved -> 'csr_vs_ml_predictions'")

# Collect to pandas for metric + plotting work (a few thousand rows per year)
preds_pdf = preds_all.toPandas()
print(f"preds_pdf shape: {preds_pdf.shape}")


StatementMeta(, d19cdaca-6fa0-468f-918e-5ca30b324312, 24, Submitted, Running, Running, True)

  Year 2023: trained on 14,518 rows  ->  predicted 5,441 rows


  Year 2024: trained on 19,959 rows  ->  predicted 5,480 rows


  Year 2025: trained on 25,439 rows  ->  predicted 5,566 rows


---
# Metrics at Every Aggregation Level

CSR vs ML, computed at **District (CPS-wide)**, **Network**, **School**, and
**Year** levels.


In [21]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Helper to build CSR vs ML comparison at different aggregation levels
# preds_pdf must contain columns: 'y', 'y_hat_CSR', 'y_hat_ML' and the grouping column (if any).

def build_comparison(preds_pdf: pd.DataFrame, group_col: str | None, label_col_name: str) -> pd.DataFrame:
    """Aggregate errors for CSR vs ML at a given level.

    Parameters
    ----------
    preds_pdf : pd.DataFrame
        Wide prediction frame with columns 'y', 'y_hat_CSR', 'y_hat_ML' and group_col (if not None).
    group_col : str or None
        Column to group by (e.g. 'NETWORK', 'YEAR', 'SCHOOL_KEY').
        If None, all rows are treated as a single group named by label_col_name.
    label_col_name : str
        Name of the label column in the output (e.g. 'District', 'Network', 'Year').
    """

    df = preds_pdf.copy()

    # If no grouping column is provided, create a single group
    if group_col is None:
        df[label_col_name] = label_col_name
        group_col = label_col_name

    groups = []

    for g, sub in df.groupby(group_col):
        y = sub["y"].values.astype(float)
        y_csr = sub["y_hat_CSR"].values.astype(float)
        y_ml = sub["y_hat_ML"].values.astype(float)

        # Filter out rows with non-positive y when computing percentage errors to avoid div-by-zero
        mask_pos = y > 0
        y_pos = y[mask_pos]
        y_csr_pos = y_csr[mask_pos]
        y_ml_pos = y_ml[mask_pos]

        def mape(y_true, y_pred):
            if len(y_true) == 0:
                return np.nan
            return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

        def medape(y_true, y_pred):
            if len(y_true) == 0:
                return np.nan
            return np.median(np.abs((y_true - y_pred) / y_true)) * 100

        mae_csr = mean_absolute_error(y, y_csr)
        mae_ml  = mean_absolute_error(y, y_ml)
        rmse_csr = mean_squared_error(y, y_csr, squared=False)
        rmse_ml  = mean_squared_error(y, y_ml, squared=False)
        r2_csr = r2_score(y, y_csr) if len(np.unique(y)) > 1 else np.nan
        r2_ml  = r2_score(y, y_ml)  if len(np.unique(y)) > 1 else np.nan

        mape_csr = mape(y_pos, y_csr_pos)
        mape_ml  = mape(y_pos, y_ml_pos)
        medape_csr = medape(y_pos, y_csr_pos)
        medape_ml  = medape(y_pos, y_ml_pos)

        mae_impr = (mae_csr - mae_ml) / mae_csr * 100 if mae_csr != 0 else np.nan
        rmse_impr = (rmse_csr - rmse_ml) / rmse_csr * 100 if rmse_csr != 0 else np.nan

        mae_winner = "ML" if mae_ml < mae_csr else ("CSR" if mae_csr < mae_ml else "Tie")
        if not np.isnan(r2_csr) and not np.isnan(r2_ml):
            r2_winner = "ML" if r2_ml > r2_csr else ("CSR" if r2_csr > r2_ml else "Tie")
        else:
            r2_winner = np.nan

        groups.append({
            label_col_name: g,
            "n": len(sub),
            "MAE_CSR": mae_csr,
            "MAE_ML": mae_ml,
            "RMSE_CSR": rmse_csr,
            "RMSE_ML": rmse_ml,
            "R2_CSR": r2_csr,
            "R2_ML": r2_ml,
            "MAPE_CSR": mape_csr,
            "MAPE_ML": mape_ml,
            "MedAPE_CSR": medape_csr,
            "MedAPE_ML": medape_ml,
            "MAE_impr_%": mae_impr,
            "RMSE_impr_%": rmse_impr,
            "MAE_winner": mae_winner,
            "R2_winner": r2_winner,
        })

    return pd.DataFrame(groups)


# -------------------------------------------------------------------------
# Existing reporting code using build_comparison
# -------------------------------------------------------------------------

pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

# ── District level (CPS-wide single aggregate) ────────────────────────────
cmp_district = build_comparison(preds_pdf, None, "District")
print("=" * 70)
print("DISTRICT LEVEL  (CPS-wide, all years pooled)")
print("=" * 70)
print(cmp_district[["District", "n",
                    "MAE_CSR", "MAE_ML", "RMSE_CSR", "RMSE_ML",
                    "R2_CSR", "R2_ML", "MAPE_CSR", "MAPE_ML",
                    "MedAPE_CSR", "MedAPE_ML"]].to_string(index=False))

# ── Network level ─────────────────────────────────────────────────────────
cmp_network = build_comparison(preds_pdf, "NETWORK", "NETWORK") \
    .sort_values("MAE_ML").reset_index(drop=True)
print("\n" + "=" * 70)
print("NETWORK LEVEL")
print("=" * 70)
print(cmp_network[["NETWORK", "n", "MAE_CSR", "MAE_ML", "MAE_impr_%",
                   "RMSE_CSR", "RMSE_ML", "MAPE_CSR", "MAPE_ML",
                   "MAE_winner"]].to_string(index=False))

# ── Year level ────────────────────────────────────────────────────────────
cmp_year = build_comparison(preds_pdf, "YEAR", "YEAR").sort_values("YEAR").reset_index(drop=True)
print("\n" + "=" * 70)
print("YEAR-WISE")
print("=" * 70)
print(cmp_year[["YEAR", "n", "MAE_CSR", "MAE_ML", "RMSE_CSR", "RMSE_ML",
                "R2_CSR", "R2_ML", "MAPE_CSR", "MAPE_ML", "MAE_winner"]].to_string(index=False))

# ── School level (one row per school; preview head + tail by ML MAE) ──────
cmp_school = build_comparison(preds_pdf, "SCHOOL_KEY", "SCHOOL_KEY") \
    .sort_values("MAE_ML").reset_index(drop=True)
print("\n" + "=" * 70)
print(f"SCHOOL LEVEL  ({len(cmp_school):,} schools)  — best & worst by ML MAE")
print("=" * 70)
school_view = cmp_school[["SCHOOL_KEY", "n", "MAE_CSR", "MAE_ML", "MAPE_CSR", "MAPE_ML", "MAE_winner"]]
print("Best 5:\n" + school_view.head(5).to_string(index=False))
print("\nWorst 5:\n" + school_view.tail(5).to_string(index=False))


StatementMeta(, , -1, Waiting, , Waiting, True)

---
## Section 20 — Task 3: Unified Comparison & Where ML Wins / Loses

A single tidy table stacks every level so CSR and ML sit side-by-side, with the
winner and improvement % per metric. Then a short verdict summarises where ML
beats CSR and where it doesn't.


In [ ]:
def tidy(cmp_df, level, key_col):
    out = cmp_df.copy()
    out.insert(0, "Level", level)
    out = out.rename(columns={key_col: "Group"})
    keep = ["Level", "Group", "n",
            "MAE_CSR", "MAE_ML", "MAE_impr_%", "MAE_winner",
            "RMSE_CSR", "RMSE_ML", "RMSE_impr_%",
            "MAPE_CSR", "MAPE_ML", "MedAPE_CSR", "MedAPE_ML",
            "R2_CSR", "R2_ML", "R2_winner"]
    return out[[c for c in keep if c in out.columns]]

unified = pd.concat([
    tidy(cmp_district, "District", "District"),
    tidy(cmp_network,  "Network",  "NETWORK"),
    tidy(cmp_year,     "Year",     "YEAR"),
], ignore_index=True)

print("UNIFIED CSR vs ML COMPARISON (District + Network + Year)")
print("=" * 100)
print(unified.to_string(index=False))

try:
    spark.createDataFrame(unified.astype({"Group": "string"})) \
        .write.mode("overwrite").option("overwriteSchema", "true") \
        .saveAsTable("csr_vs_ml_comparison")
    print("\nSaved -> 'csr_vs_ml_comparison'")
except Exception as e:
    print(f"\n(Comparison table not saved: {e})")


# ── Verdict ───────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("VERDICT — where ML beats CSR")
print("=" * 70)
d = cmp_district.iloc[0]
print(f"District-wide: MAE {d['MAE_CSR']:.1f} -> {d['MAE_ML']:.1f} "
      f"({d['MAE_impr_%']:+.1f}%),  RMSE {d['RMSE_CSR']:.1f} -> {d['RMSE_ML']:.1f} "
      f"({d['RMSE_impr_%']:+.1f}%),  MedAPE {d['MedAPE_CSR']:.1f}% -> {d['MedAPE_ML']:.1f}%")

net_ml_wins = (cmp_network["MAE_winner"] == "ML").sum()
print(f"\nNetworks where ML wins on MAE : {net_ml_wins}/{len(cmp_network)}")
losers = cmp_network[cmp_network["MAE_winner"] == "CSR"]
if len(losers):
    print("  Networks where CSR still wins (MAE):")
    print(losers[["NETWORK", "MAE_CSR", "MAE_ML"]].to_string(index=False))
else:
    print("  ML wins on MAE in every network.")

yr_ml_wins = (cmp_year["MAE_winner"] == "ML").sum()
print(f"\nYears where ML wins on MAE     : {yr_ml_wins}/{len(cmp_year)}")
sch_ml_wins = (cmp_school["MAE_winner"] == "ML").mean() * 100
print(f"Schools where ML wins on MAE   : {sch_ml_wins:.1f}%")

# print("\nNote: CSR's RMSE is inflated by a few schools where AVG_SURVIVAL_RATE_3YR")
# print("blows up (feeder grade near 0). MedAPE is the fairer headline — it down-")
# print("weights those outliers and still favours ML.")


StatementMeta(, , -1, Waiting, , Waiting, True)

---
## Section 21 — Task 4: Visualizations

1. District-level grouped bars (MAE / RMSE / MAPE / R²)
2. Network-level grouped bars (MAE and MedAPE)
3. Year-wise trend lines (MAE, RMSE, MedAPE)
4. Actual vs predicted scatter (ML vs CSR)
5. Per-school error-improvement distribution


In [ ]:
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
CSR_C, ML_C = "#d1495b", "#2e86ab"

# ── 1. District-level grouped bars ────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, m in zip(axes, ["MAE", "RMSE", "MAPE", "R2"]):
    vals = [cmp_district.iloc[0][f"{m}_CSR"], cmp_district.iloc[0][f"{m}_ML"]]
    bars = ax.bar(["CSR", "ML"], vals, color=[CSR_C, ML_C])
    ax.set_title(f"District {m}")
    ax.bar_label(bars, fmt="%.2f", padding=2)
axes[2].set_ylabel("%")
fig.suptitle("District-wide CSR vs ML", fontweight="bold")
fig.tight_layout(); plt.show()

# ── 2. Network-level grouped bars (MAE, MedAPE) ───────────────────────────
net = cmp_network.sort_values("MAE_ML")
x = np.arange(len(net)); w = 0.4
fig, axes = plt.subplots(2, 1, figsize=(max(8, len(net) * 0.7), 9))
for ax, m in zip(axes, ["MAE", "MedAPE"]):
    ax.bar(x - w/2, net[f"{m}_CSR"], w, label="CSR", color=CSR_C)
    ax.bar(x + w/2, net[f"{m}_ML"],  w, label="ML",  color=ML_C)
    ax.set_xticks(x); ax.set_xticklabels(net["NETWORK"], rotation=45, ha="right")
    ax.set_title(f"{m} by Network"); ax.set_ylabel(m + (" (%)" if "APE" in m else ""))
    ax.legend()
fig.suptitle("Network-level CSR vs ML", fontweight="bold")
fig.tight_layout(); plt.show()

# ── 3. Year-wise trend lines ──────────────────────────────────────────────
yr = cmp_year.sort_values("YEAR")
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
for ax, m in zip(axes, ["MAE", "RMSE", "MedAPE"]):
    ax.plot(yr["YEAR"], yr[f"{m}_CSR"], "o-", color=CSR_C, label="CSR")
    ax.plot(yr["YEAR"], yr[f"{m}_ML"],  "s-", color=ML_C,  label="ML")
    ax.set_title(f"{m} over time"); ax.set_xlabel("Year")
    ax.set_ylabel(m + (" (%)" if "APE" in m else "")); ax.legend()
    ax.set_xticks(yr["YEAR"])
fig.suptitle("Year-wise CSR vs ML performance", fontweight="bold")
fig.tight_layout(); plt.show()


StatementMeta(, , -1, Waiting, , Waiting, True)

---
## Section 19 — Grade-Wise CSR vs ML Comparison

The same out-of-sample backtest (`preds_pdf`) and `build_comparison()` helper, now
grouped by **GRADE**. This shows where the ML model helps most across the grade
span — and, for any entry grade present in the data, how each method copes when the
feeder signal is weakest. Grades are ordered by `GRADE_ORDER`, not alphabetically.

In [ ]:
# ── Grade-wise CSR vs ML error comparison (out-of-sample, all backtest years) ──
cmp_grade = build_comparison(preds_pdf, "GRADE", "GRADE")
cmp_grade["_order"] = cmp_grade["GRADE"].map(lambda g: GRADE_ORDER.get(g, 99))
cmp_grade = cmp_grade.sort_values("_order").drop(columns="_order").reset_index(drop=True)

print("=" * 84)
print("GRADE-WISE  (out-of-sample, all backtest years pooled)")
print("=" * 84)
print(cmp_grade[["GRADE", "n", "MAE_CSR", "MAE_ML", "MAE_impr_%",
                 "RMSE_CSR", "RMSE_ML", "MAPE_CSR", "MAPE_ML",
                 "MedAPE_CSR", "MedAPE_ML", "MAE_winner"]].to_string(index=False))

g_ml_wins = (cmp_grade["MAE_winner"] == "ML").sum()
print(f"\nGrades where ML wins on MAE: {g_ml_wins}/{len(cmp_grade)}")

# Add the grade level to the unified comparison family and persist for the dashboard
unified_grade = tidy(cmp_grade, "Grade", "GRADE")
try:
    spark.createDataFrame(unified_grade.astype({"Group": "string"})) \
        .write.mode("overwrite").option("overwriteSchema", "true") \
        .saveAsTable("csr_vs_ml_comparison_by_grade")
    print("Saved -> 'csr_vs_ml_comparison_by_grade'")
except Exception as e:
    print(f"(Grade comparison table not saved: {e})")

In [ ]:
# ── Grade-wise CSR vs ML bar charts (matches the Section 18 plot style) ──────
g = cmp_grade.copy()
x = np.arange(len(g)); w = 0.4
fig, axes = plt.subplots(2, 1, figsize=(max(8, len(g) * 0.7), 9))
for ax, m in zip(axes, ["MAE", "MedAPE"]):
    ax.bar(x - w/2, g[f"{m}_CSR"], w, label="CSR", color=CSR_C)
    ax.bar(x + w/2, g[f"{m}_ML"],  w, label="ML",  color=ML_C)
    ax.set_xticks(x)
    ax.set_xticklabels(["Grade " + str(gr) for gr in g["GRADE"]], rotation=45, ha="right")
    ax.set_title(f"{m} by Grade")
    ax.set_ylabel(m + (" (%)" if "APE" in m else ""))
    ax.legend()
fig.suptitle("Grade-wise CSR vs ML performance", fontweight="bold")
fig.tight_layout(); plt.show()